# Colab Runner For Multi-Agent Reward Verification

This notebook is a Colab-first entrypoint for the repository.

It is configured to:
- install the project dependencies in Colab
- locate or optionally clone the repository into `/content`
- run smaller train/test experiments that fit a normal Colab session
- inspect the strict reward verification pipeline

Default backend: `heuristic`

That default is intentional. Standard Colab does not ship with Ollama running locally, so the heuristic backend is the cleanest way to run the full pipeline end-to-end.


In [ ]:
import subprocess
import sys

# Install runtime dependencies.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets>=2.19.0',
    'pyyaml>=6.0',
    'matplotlib>=3.8.0',
    'pandas>=2.0.0',
    'numpy>=1.26.0',
])


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Optional: set this if you want the notebook to clone the repo automatically.
REPO_URL = ""
CLONE_DIR = Path('/content/multi-agent-rl-environment')

if REPO_URL and not CLONE_DIR.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(CLONE_DIR)])

def locate_project_root() -> Path:
    candidates = [Path.cwd(), Path('/content'), CLONE_DIR]
    for base in candidates:
        if not base.exists():
            continue
        direct = base
        if (direct / 'pyproject.toml').exists() and (direct / 'environment').exists():
            return direct
        for child in base.iterdir():
            if child.is_dir() and (child / 'pyproject.toml').exists() and (child / 'environment').exists():
                return child
    raise FileNotFoundError(
        'Could not find the project root. Upload the repo to Colab or set REPO_URL above.'
    )

ROOT = locate_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('project root:', ROOT)


In [ ]:
from environment import MultiAgentEnvironment
from experiment import ExperimentRunner
from run_demo import (
    build_heuristic_agents,
    build_icl_ollama_agents,
    build_ollama_agents,
    DEFAULT_OLLAMA_MODELS,
)
from agents import build_self_refine_ollama_agents
from run_experiment import chunk_tasks, iter_tasks


def build_agents(cfg):
    backend = cfg['backend']
    if backend == 'heuristic':
        return build_heuristic_agents(use_rl=cfg.get('use_rl', False))
    if backend == 'ollama':
        return build_ollama_agents(cfg['models'], cfg.get('ollama_host'), cfg['seed'])
    if backend == 'icl':
        return build_icl_ollama_agents(
            models=cfg['models'],
            host=cfg.get('ollama_host'),
            seed=cfg['seed'],
            memory_strategy=cfg.get('icl_strategy', 'reward_weighted'),
            prompt_memory_size=int(cfg.get('prompt_memory_size', 5)),
            memory_buffer_size=int(cfg.get('memory_buffer_size', 128)),
            oracle_gt_threshold=float(cfg.get('oracle_gt_threshold', 0.99)),
            include_eval_memory_in_prompt=bool(cfg.get('include_eval_memory_in_prompt', True)),
        )
    if backend == 'self_refine':
        return build_self_refine_ollama_agents(cfg['models'], cfg.get('ollama_host'), cfg['seed'])
    raise ValueError(f"Unknown backend: {backend}")


def run_pipeline(cfg):
    task_iter = iter_tasks(
        dataset_name=cfg['dataset'],
        split=cfg['split'],
        limit=cfg['limit'],
        seed=cfg['seed'],
        start_index=cfg['start_index'],
        streaming=cfg.get('streaming', False),
    )
    env = MultiAgentEnvironment(
        agents=build_agents(cfg),
        alpha=cfg['alpha'],
        task_type=cfg.get('task_type', 'flexible'),
        seed=cfg['seed'],
        max_concurrency=cfg.get('max_concurrency'),
        revision_rounds=cfg.get('revision_rounds', 0),
        continue_on_agent_error=cfg.get('continue_on_agent_error', True),
        use_trust_weighting=cfg.get('use_trust_weighting', True),
        historical_trust_blend=cfg.get('historical_trust_blend', 0.5),
        trust_floor=cfg.get('trust_floor', 0.1),
        use_attention=cfg.get('use_attention', True),
        attention_top_k=cfg.get('attention_top_k', 2),
        attention_temperature=cfg.get('attention_temperature', 1.0),
        attention_entropy_coef=cfg.get('attention_entropy_coef', 0.0),
        use_rl=cfg.get('use_rl', False),
    )
    output_dir = ROOT / cfg['output_dir']
    runner = ExperimentRunner(env=env, output_dir=output_dir, run_manifest=cfg)
    result = None
    for batch in chunk_tasks(task_iter, cfg['batch_size']):
        result = runner.run(
            batch,
            resume=cfg.get('resume', False),
            apply_updates=cfg.get('apply_updates', False),
            track_learning_curve=cfg.get('track_learning_curve', True),
        )
    if result is None:
        result = runner.run(
            [],
            resume=cfg.get('resume', False),
            apply_updates=cfg.get('apply_updates', False),
            track_learning_curve=cfg.get('track_learning_curve', True),
        )
    return result


def inspect_reward_verification(record):
    verification = record['metadata']['reward_verification']
    return {
        'reward_formula': record['metadata'].get('reward_formula'),
        'stage_order': verification['stage_order'],
        'sanity_checks': verification['sanity_checks'],
        'ground_truth_reward': verification['ground_truth_reward'],
        'combined_peer_reward': verification['combined_peer_reward'],
        'final_reward': verification['final_reward'],
    }


## Colab Train Configuration

The defaults below are intentionally conservative for Colab:
- `heuristic` backend by default
- smaller limits than the local notebook
- no resume by default
- trust and attention enabled


In [ ]:
COLAB_TRAIN_CONFIG = {
    'dataset': 'gsm8k',
    'split': 'train',
    'backend': 'heuristic',
    'models': DEFAULT_OLLAMA_MODELS,
    'ollama_host': None,
    'seed': 7,
    'limit': 20,
    'start_index': 0,
    'batch_size': 5,
    'alpha': 0.8,
    'task_type': 'flexible',
    'apply_updates': True,
    'resume': False,
    'track_learning_curve': True,
    'streaming': True,
    'max_concurrency': 2,
    'revision_rounds': 1,
    'continue_on_agent_error': True,
    'use_trust_weighting': True,
    'historical_trust_blend': 0.5,
    'trust_floor': 0.1,
    'use_attention': True,
    'attention_top_k': 2,
    'attention_temperature': 1.0,
    'attention_entropy_coef': 0.0,
    'use_rl': False,
    'icl_strategy': 'reward_weighted',
    'prompt_memory_size': 5,
    'memory_buffer_size': 128,
    'output_dir': 'outputs/colab-train',
}

COLAB_TRAIN_CONFIG


In [ ]:
# Run this cell to execute the Colab train-style experiment.
colab_train_result = run_pipeline(COLAB_TRAIN_CONFIG)
colab_train_result


## Colab Test Configuration

Use this for a smaller frozen evaluation pass after a train-style run.


In [ ]:
COLAB_TEST_CONFIG = {
    **COLAB_TRAIN_CONFIG,
    'split': 'test',
    'limit': 10,
    'apply_updates': False,
    'output_dir': 'outputs/colab-test',
}

COLAB_TEST_CONFIG


In [ ]:
# Run this cell to execute the Colab test-style evaluation.
colab_test_result = run_pipeline(COLAB_TEST_CONFIG)
colab_test_result


## Inspect Summary


In [ ]:
def load_summary(output_dir):
    path = ROOT / output_dir / 'summary.json'
    if not path.exists():
        print(f'No summary file found yet: {path}')
        return None
    return json.loads(path.read_text(encoding='utf-8'))

summary = load_summary(COLAB_TEST_CONFIG['output_dir'])
summary.keys() if summary else None


In [ ]:
if summary:
    try:
        import pandas as pd
        display(pd.DataFrame(summary['leaderboard']))
        display(pd.DataFrame(summary['agent_metrics']).T)
        verification_metrics = summary.get('verification_metrics') or {}
        if verification_metrics:
            display(pd.DataFrame([verification_metrics]))
    except ImportError:
        print(json.dumps(summary, indent=2))
else:
    print('Run an experiment first to populate summary.json.')


## Inspect Reward Verification


In [ ]:
results_path = ROOT / COLAB_TEST_CONFIG['output_dir'] / 'results.jsonl'
if results_path.exists():
    rows = [json.loads(line) for line in results_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('num rows:', len(rows))
    rows[0]['metadata'].keys() if rows else None
else:
    rows = []
    print(f'No results file found yet: {results_path}')


In [ ]:
if rows:
    verification = rows[0]['metadata']['reward_verification']
    print(rows[0]['metadata'].get('reward_formula'))
    print('stage order:', verification['stage_order'])
    print('sanity checks:', verification['sanity_checks'])
    try:
        import pandas as pd
        display(pd.DataFrame(verification['ground_truth_reward']))
        display(pd.DataFrame(verification['score_normalization']))
        display(pd.DataFrame(verification['trust_weighting']['evaluators']))
        display(pd.DataFrame(verification['combined_peer_reward']))
        display(pd.DataFrame(verification['final_reward']))
    except ImportError:
        print(json.dumps(inspect_reward_verification(rows[0]), indent=2))
else:
    print('Run an experiment first to inspect reward verification.')
